In [1]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Date-Time-Demo'). \
    master('yarn'). \
    getOrCreate()

In [4]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Date-Time-Demo'). \
#     master('local[*]'). \
#     getOrCreate()

In [5]:
sc = spark.sparkContext

## Deep dive into Apache Spark DateTime Functions

DateTime functions will always be tricky but very important irrespective of language or framework. In this Notebook, we review the DateTime functions available in Apache Spark™️. We show examples in an easy to comprehend standardized format to augment the brief examples in the Apache Spark docs.

### Setup

Here are a few key setup steps before trying the examples further down.

In [6]:
from pyspark.sql.functions import *

#### Create Sample DataFrame

Let’s try to create a sample DataFrame so that we can use it for the rest of this Notebook to understand the various DateTime functions.

In [7]:
emp = [(1, "AAA", "dept1", 1000, "2019-02-01 15:12:13"),
    (2, "BBB", "dept1", 1100, "2018-04-01 5:12:3"),
    (3, "CCC", "dept1", 3000, "2017-06-05 1:2:13"),
    (4, "DDD", "dept1", 1500, "2019-08-10 10:52:53"),
    (5, "EEE", "dept2", 8000, "2016-01-11 5:52:43"),
    (6, "FFF", "dept2", 7200, "2015-04-14 19:32:33"),
    (7, "GGG", "dept3", 7100, "2019-02-21 15:42:43"),
    (8, "HHH", "dept3", 3700, "2016-09-25 15:32:33"),
    (9, "III", "dept3", 4500, "2017-10-15 15:22:23"),
    (10, "JJJ", "dept5", 3400, "2018-12-17 15:14:17")]

empdf = spark.createDataFrame(emp, ["id", "name", "dept", "salary", "date"])

empdf.show()
empdf.printSchema()

+---+----+-----+------+-------------------+
| id|name| dept|salary|               date|
+---+----+-----+------+-------------------+
|  1| AAA|dept1|  1000|2019-02-01 15:12:13|
|  2| BBB|dept1|  1100|  2018-04-01 5:12:3|
|  3| CCC|dept1|  3000|  2017-06-05 1:2:13|
|  4| DDD|dept1|  1500|2019-08-10 10:52:53|
|  5| EEE|dept2|  8000| 2016-01-11 5:52:43|
|  6| FFF|dept2|  7200|2015-04-14 19:32:33|
|  7| GGG|dept3|  7100|2019-02-21 15:42:43|
|  8| HHH|dept3|  3700|2016-09-25 15:32:33|
|  9| III|dept3|  4500|2017-10-15 15:22:23|
| 10| JJJ|dept5|  3400|2018-12-17 15:14:17|
+---+----+-----+------+-------------------+

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- date: string (nullable = true)



#### 1. add_months

This function adds months to a date. It will return a new date, however many months from the start date. In the below statement we add 1 month to the column “date” and generated a new column as “next_month”.

In [8]:
df = (empdf
    .select("date")
    .withColumn("next_month", add_months("date", 1)))
df.show()

+-------------------+----------+
|               date|next_month|
+-------------------+----------+
|2019-02-01 15:12:13|2019-03-01|
|  2018-04-01 5:12:3|2018-05-01|
|  2017-06-05 1:2:13|2017-07-05|
|2019-08-10 10:52:53|2019-09-10|
| 2016-01-11 5:52:43|2016-02-11|
|2015-04-14 19:32:33|2015-05-14|
|2019-02-21 15:42:43|2019-03-21|
|2016-09-25 15:32:33|2016-10-25|
|2017-10-15 15:22:23|2017-11-15|
|2018-12-17 15:14:17|2019-01-17|
+-------------------+----------+



#### 2. current_date

This function returns the current date.

In [10]:
df = (empdf
    .withColumn("current_date", current_date())
    .select("id", "current_date"))
df.show(2)

+---+------------+
| id|current_date|
+---+------------+
|  1|  2026-11-02|
|  2|  2026-11-02|
+---+------------+
only showing top 2 rows



#### 3. current_timestamp

This function returns the current timestamp.

In [11]:
df = (empdf
    .withColumn("current_timestamp", current_timestamp())
    .select("id", "current_timestamp"))
df.show(2,False)       

+---+-----------------------+
|id |current_timestamp      |
+---+-----------------------+
|1  |2026-11-02 05:49:49.622|
|2  |2026-11-02 05:49:49.622|
+---+-----------------------+
only showing top 2 rows



#### 4. date_add

This function returns a date x days after the start date passed to the function. In the example below, it returns a date 5 days after “date” in a new column as “next_date”. E.g. for date: 1st Feb 2019 it returns 6th Feb 2019.

In [11]:
df = (empdf
    .select("date")
    .withColumn("next_date", date_add("date", 5)))
df.show(2)

+-------------------+----------+
|               date| next_date|
+-------------------+----------+
|2019-02-01 15:12:13|2019-02-06|
|  2018-04-01 5:12:3|2018-04-06|
+-------------------+----------+
only showing top 2 rows



#### 5. date_format

This function will convert the date to the specified format. For example, we can convert the date from “yyyy-MM-dd” to “dd/MM/yyyy” format.

In [17]:
df = (empdf
    .select("date")
    .withColumn("new_date", date_format("date", "dd/MM/yyyy")))
df.show(2)

df = (empdf
    .select("date")
    .withColumn("new_date", date_format("date", "dd/MM/yyyy HH:mm:ss")))
df.show(2)

+-------------------+----------+
|               date|  new_date|
+-------------------+----------+
|2019-02-01 15:12:13|01/02/2019|
|  2018-04-01 5:12:3|01/04/2018|
+-------------------+----------+
only showing top 2 rows

+-------------------+-------------------+
|               date|           new_date|
+-------------------+-------------------+
|2019-02-01 15:12:13|01/02/2019 15:12:13|
|  2018-04-01 5:12:3|01/04/2018 05:12:03|
+-------------------+-------------------+
only showing top 2 rows



#### 6. date_sub

This function returns a date some number of the days before the date passed to it. It is the opposite of __*date_add*__. In the example below, it returns a date that is 5 days earlier in a column as “new_date”. For example, date 1st Feb 2019 returns 27th Jan 2019.

In [18]:
df = (empdf
    .select("date")
    .withColumn("new_date", date_sub("date", 5)))
df.show(2)

+-------------------+----------+
|               date|  new_date|
+-------------------+----------+
|2019-02-01 15:12:13|2019-01-27|
|  2018-04-01 5:12:3|2018-03-27|
+-------------------+----------+
only showing top 2 rows



#### 7. date_trunc

This function returns a timestamp truncated to the specified unit. It could be a year, month, day, hour, minute, second, week or quarter.  

* **Let’s truncate the date by a year.** we can use “yyyy” or “yy” or” “year” to specify year. For timestamp “2019–02–01 15:12:13”, if we truncate based on the year it will return “2019–01–01 00:00:00”

In [30]:
df = (empdf
    .select("date")
    .withColumn("new_date", date_trunc("year", "date")))
df.show(5)

+-------------------+-------------------+
|               date|           new_date|
+-------------------+-------------------+
|2019-02-01 15:12:13|2019-01-01 00:00:00|
|  2018-04-01 5:12:3|2018-01-01 00:00:00|
|  2017-06-05 1:2:13|2017-01-01 00:00:00|
|2019-08-10 10:52:53|2019-01-01 00:00:00|
| 2016-01-11 5:52:43|2016-01-01 00:00:00|
+-------------------+-------------------+
only showing top 5 rows



Let’s truncate date by month. We use “mm” or “month” or” “mon” to specify month. For timestamp “2019–02–01 15:12:13”, if we truncate based on month it returns “2019–02–01 00:00:00”

In [31]:
df = (empdf
    .select("date")
    .withColumn("new_date", date_trunc("month", "date")))
df.show(5)

+-------------------+-------------------+
|               date|           new_date|
+-------------------+-------------------+
|2019-02-01 15:12:13|2019-02-01 00:00:00|
|  2018-04-01 5:12:3|2018-04-01 00:00:00|
|  2017-06-05 1:2:13|2017-06-01 00:00:00|
|2019-08-10 10:52:53|2019-08-01 00:00:00|
| 2016-01-11 5:52:43|2016-01-01 00:00:00|
+-------------------+-------------------+
only showing top 5 rows



Let’s truncate date by day. We can use “day” or “dd” to specify day. For timestamp “2019–02–10 15:12:13”, if we truncate based on day it will return “2019–02–10 00:00:00”

In [32]:
df = (empdf
    .select("date")
    .withColumn("new_date", date_trunc("day", "date")))
df.show()

+-------------------+-------------------+
|               date|           new_date|
+-------------------+-------------------+
|2019-02-01 15:12:13|2019-02-01 00:00:00|
|  2018-04-01 5:12:3|2018-04-01 00:00:00|
|  2017-06-05 1:2:13|2017-06-05 00:00:00|
|2019-08-10 10:52:53|2019-08-10 00:00:00|
| 2016-01-11 5:52:43|2016-01-11 00:00:00|
|2015-04-14 19:32:33|2015-04-14 00:00:00|
|2019-02-21 15:42:43|2019-02-21 00:00:00|
|2016-09-25 15:32:33|2016-09-25 00:00:00|
|2017-10-15 15:22:23|2017-10-15 00:00:00|
|2018-12-17 15:14:17|2018-12-17 00:00:00|
+-------------------+-------------------+



**Note:** we can use these many formats to truncate the date based on different levels. we can use these formats: ‘year’, ‘yyyy’, ‘yy’, ‘month’, ‘mon’, ‘mm’, ‘day’, ‘dd’, ‘hour’, ‘minute’, ‘second’, ‘week’, ‘quarter’.

#### 8. datediff

This function returns the difference between dates in terms of days. Let’s add another column as the current date and then take the difference between “current_date” and “date”.

In [33]:
df = (empdf.select("date")
        # Add another date column as current date.
        .withColumn("current_date", current_date()) 
        # Take the difference between current_date and date column.
        .withColumn("date_diff", datediff("current_date", "date"))) 
df.show(2)

+-------------------+------------+---------+
|               date|current_date|date_diff|
+-------------------+------------+---------+
|2019-02-01 15:12:13|  2026-06-05|     2681|
|  2018-04-01 5:12:3|  2026-06-05|     2987|
+-------------------+------------+---------+
only showing top 2 rows



#### 9. dayofmonth

This function returns the day of the month. For 5th Jan 2019 (2019–01–05) it will return 5.

In [35]:
df = (empdf
    .select("date")
    .withColumn("dayofmonth", dayofmonth("date")))
df.show()

+-------------------+----------+
|               date|dayofmonth|
+-------------------+----------+
|2019-02-01 15:12:13|         1|
|  2018-04-01 5:12:3|         1|
|  2017-06-05 1:2:13|         5|
|2019-08-10 10:52:53|        10|
| 2016-01-11 5:52:43|        11|
|2015-04-14 19:32:33|        14|
|2019-02-21 15:42:43|        21|
|2016-09-25 15:32:33|        25|
|2017-10-15 15:22:23|        15|
|2018-12-17 15:14:17|        17|
+-------------------+----------+



#### 10. dayofweek

This function returns the day of the week as an integer. It will consider Sunday as 1st and Saturday as 7th. For 1st Feb 2019 (2019–02–01) which is Friday, it will return 6. Similarly, for 1st April 2018 (2018–04–01) which is Sunday, it will return 1.

In [36]:
df = (empdf
    .select("date")
    .withColumn("dayofweek", dayofweek("date")))
df.show(2)

+-------------------+---------+
|               date|dayofweek|
+-------------------+---------+
|2019-02-01 15:12:13|        6|
|  2018-04-01 5:12:3|        1|
+-------------------+---------+
only showing top 2 rows



#### 11. dayofyear

This function returns the day of the year as an integer. For 1st Feb it will return 32 (31 days of Jan +1 day of Feb). For 1st April 2018, it will return 91 (31 days of Jan + 28 days of Feb (2018 is a non-leap year) + 31 days of Mar + 1 day of April).

In [6]:
df = (empdf
    .select("date")
    .withColumn("dayofyear", dayofyear("date")))
df.show(2)

+-------------------+---------+
|               date|dayofyear|
+-------------------+---------+
|2019-02-01 15:12:13|       32|
|  2018-04-01 5:12:3|       91|
+-------------------+---------+
only showing top 2 rows



#### 12. from_utc_timestamp

This function converts UTC timestamps to timestamps of any specified timezone. By default, it assumes the date is a UTC timestamp.
Let's convert a UTC timestamp to “PST” time.

In [14]:
df = (empdf
    .select("date")
    .withColumn("pst_timestamp", from_utc_timestamp("date", "PST")))
df.show()

print(spark.conf.get("spark.sql.session.timeZone"))
df = (empdf
    .select("date")
    .withColumn("ist_timestamp", from_utc_timestamp("date", "Asia/Kolkata")))
df.show()

+-------------------+-------------------+
|               date|      pst_timestamp|
+-------------------+-------------------+
|2019-02-01 15:12:13|2019-02-01 07:12:13|
|  2018-04-01 5:12:3|2018-03-31 22:12:03|
|  2017-06-05 1:2:13|2017-06-04 18:02:13|
|2019-08-10 10:52:53|2019-08-10 03:52:53|
| 2016-01-11 5:52:43|2016-01-10 21:52:43|
|2015-04-14 19:32:33|2015-04-14 12:32:33|
|2019-02-21 15:42:43|2019-02-21 07:42:43|
|2016-09-25 15:32:33|2016-09-25 08:32:33|
|2017-10-15 15:22:23|2017-10-15 08:22:23|
|2018-12-17 15:14:17|2018-12-17 07:14:17|
+-------------------+-------------------+

Asia/Kolkata
+-------------------+-------------------+
|               date|      ist_timestamp|
+-------------------+-------------------+
|2019-02-01 15:12:13|2019-02-01 20:42:13|
|  2018-04-01 5:12:3|2018-04-01 10:42:03|
|  2017-06-05 1:2:13|2017-06-05 06:32:13|
|2019-08-10 10:52:53|2019-08-10 16:22:53|
| 2016-01-11 5:52:43|2016-01-11 11:22:43|
|2015-04-14 19:32:33|2015-04-15 01:02:33|
|2019-02-21 15:42:43

#### 13. unix_timestamp

This function converts timestamp strings of the given format to Unix timestamps (in seconds). The default format is “yyyy-MM-dd HH:mm:ss”. (Note: You can use spark property: “spark.sql.session.timeZone” to set the timezone.)

In [11]:
df = (empdf
    .select("date")
    .withColumn("unix_timestamp", unix_timestamp("date", "yyyy-MM-dd HH:mm:ss")))
df.show(2)

+-------------------+--------------+
|               date|unix_timestamp|
+-------------------+--------------+
|2019-02-01 15:12:13|    1549014133|
|  2018-04-01 5:12:3|    1522539723|
+-------------------+--------------+
only showing top 2 rows



#### 14. from_unixtime

This function converts the number of seconds from Unix epoch (1970–01–01 00:00:00 UTC) to a given string format. You can set the timezone and format as well. (Note: you can use spark property: “spark.sql.session.timeZone” to set the timezone). For demonstration purposes, we have converted the timestamp to Unix timestamp and converted it back to timestamp.

In [15]:
df = (empdf
    .select("date")
    # Convert timestamp to unix timestamp.
    .withColumn("unix_timestamp", unix_timestamp("date", "yyyy-MM-dd HH:mm:ss"))
    # Convert unix timestamp to timestamp.
    .withColumn("date_from_unixtime", from_unixtime("unix_timestamp")))
df.show()

+-------------------+--------------+-------------------+
|               date|unix_timestamp| date_from_unixtime|
+-------------------+--------------+-------------------+
|2019-02-01 15:12:13|    1549014133|2019-02-01 15:12:13|
|  2018-04-01 5:12:3|    1522539723|2018-04-01 05:12:03|
|  2017-06-05 1:2:13|    1496604733|2017-06-05 01:02:13|
|2019-08-10 10:52:53|    1565414573|2019-08-10 10:52:53|
| 2016-01-11 5:52:43|    1452471763|2016-01-11 05:52:43|
|2015-04-14 19:32:33|    1429020153|2015-04-14 19:32:33|
|2019-02-21 15:42:43|    1550743963|2019-02-21 15:42:43|
|2016-09-25 15:32:33|    1474797753|2016-09-25 15:32:33|
|2017-10-15 15:22:23|    1508061143|2017-10-15 15:22:23|
|2018-12-17 15:14:17|    1545039857|2018-12-17 15:14:17|
+-------------------+--------------+-------------------+



#### 15. hour

In [6]:
# It will return hour part of the date.
df = (empdf
    .select("date")
    .withColumn("hour", hour("date")))
df.show()

+-------------------+----+
|               date|hour|
+-------------------+----+
|2019-02-01 15:12:13|  15|
|  2018-04-01 5:12:3|   5|
|  2017-06-05 1:2:13|   1|
|2019-08-10 10:52:53|  10|
| 2016-01-11 5:52:43|   5|
|2015-04-14 19:32:33|  19|
|2019-02-21 15:42:43|  15|
|2016-09-25 15:32:33|  15|
|2017-10-15 15:22:23|  15|
|2018-12-17 15:14:17|  15|
+-------------------+----+



#### 16. last_day

In [8]:
# It will return last of the Month for a given date.
# For e.g. for 5th Jan 2019, it will return 31st Jan 2019, since this is the last date for the Month.

df = empdf.select("date").withColumn("last_date", last_day("date"))
df.show()

+-------------------+----------+
|               date| last_date|
+-------------------+----------+
|2019-02-01 15:12:13|2019-02-28|
|  2018-04-01 5:12:3|2018-04-30|
|  2017-06-05 1:2:13|2017-06-30|
|2019-08-10 10:52:53|2019-08-31|
| 2016-01-11 5:52:43|2016-01-31|
|2015-04-14 19:32:33|2015-04-30|
|2019-02-21 15:42:43|2019-02-28|
|2016-09-25 15:32:33|2016-09-30|
|2017-10-15 15:22:23|2017-10-31|
|2018-12-17 15:14:17|2018-12-31|
+-------------------+----------+



#### 17. minute

In [9]:
# It will return minute part of the date.
df = (empdf
    .select("date")
    .withColumn("minute", minute("date")))
df.show()

+-------------------+------+
|               date|minute|
+-------------------+------+
|2019-02-01 15:12:13|    12|
|  2018-04-01 5:12:3|    12|
|  2017-06-05 1:2:13|     2|
|2019-08-10 10:52:53|    52|
| 2016-01-11 5:52:43|    52|
|2015-04-14 19:32:33|    32|
|2019-02-21 15:42:43|    42|
|2016-09-25 15:32:33|    32|
|2017-10-15 15:22:23|    22|
|2018-12-17 15:14:17|    14|
+-------------------+------+



#### 18. month

In [11]:
# It will return month part of the date.
df = (empdf
    .select("date")
    .withColumn("month", month("date")))
df.show()

+-------------------+-----+
|               date|month|
+-------------------+-----+
|2019-02-01 15:12:13|    2|
|  2018-04-01 5:12:3|    4|
|  2017-06-05 1:2:13|    6|
|2019-08-10 10:52:53|    8|
| 2016-01-11 5:52:43|    1|
|2015-04-14 19:32:33|    4|
|2019-02-21 15:42:43|    2|
|2016-09-25 15:32:33|    9|
|2017-10-15 15:22:23|   10|
|2018-12-17 15:14:17|   12|
+-------------------+-----+



#### 19. months_between

In [15]:
# It will return the difference between the dates in terms of months. 
# If first date is greater than second one result will be positive else negative.
# For e.g. between 6th Feb 2019 and 5th Jan 2019 it will return 1.

df = (empdf
    .select("date")
    # Add another date column as current date.        
    .withColumn("current_date", current_date()) 
    # Take the difference between current_date and date column in terms of months.
    .withColumn("months_between", months_between("current_date", "date"))) 
df.show()

# Note from Spark 2.4.0 onwards you can specify third argument "roundOff=True" to round-Off the value. 
# Default value is True.



+-------------------+------------+--------------+
|               date|current_date|months_between|
+-------------------+------------+--------------+
|2019-02-01 15:12:13|  2026-06-10|   88.26988762|
|  2018-04-01 5:12:3|  2026-06-10|   98.28333221|
|  2017-06-05 1:2:13|  2026-06-10|  108.15989658|
|2019-08-10 10:52:53|  2026-06-10|          82.0|
| 2016-01-11 5:52:43|  2026-06-10|  124.95984058|
|2015-04-14 19:32:33|  2026-06-10|  133.84470094|
|2019-02-21 15:42:43|  2026-06-10|   87.62404309|
|2016-09-25 15:32:33|  2026-06-10|  116.49523858|
|2017-10-15 15:22:23|  2026-06-10|  103.81804697|
|2018-12-17 15:14:17|  2026-06-10|   89.75371229|
+-------------------+------------+--------------+



#### 20. next_day

In [16]:
# It will return the next day based on the dayOfWeek specified in next argument.
# For e.g. for 1st Feb 2019 (Friday) if we ask for next_day as sunday, it will return 3rd Feb 2019.

df = (empdf
    .select("date")
    .withColumn("next_day", next_day("date", "sun")))
df.show()

+-------------------+----------+
|               date|  next_day|
+-------------------+----------+
|2019-02-01 15:12:13|2019-02-03|
|  2018-04-01 5:12:3|2018-04-08|
|  2017-06-05 1:2:13|2017-06-11|
|2019-08-10 10:52:53|2019-08-11|
| 2016-01-11 5:52:43|2016-01-17|
|2015-04-14 19:32:33|2015-04-19|
|2019-02-21 15:42:43|2019-02-24|
|2016-09-25 15:32:33|2016-10-02|
|2017-10-15 15:22:23|2017-10-22|
|2018-12-17 15:14:17|2018-12-23|
+-------------------+----------+



#### 21. quarter

In [17]:
# It will return quarter of the given date as integer.

df = (empdf
    .select("date")
    .withColumn("quarter", quarter("date")))
df.show()

+-------------------+-------+
|               date|quarter|
+-------------------+-------+
|2019-02-01 15:12:13|      1|
|  2018-04-01 5:12:3|      2|
|  2017-06-05 1:2:13|      2|
|2019-08-10 10:52:53|      3|
| 2016-01-11 5:52:43|      1|
|2015-04-14 19:32:33|      2|
|2019-02-21 15:42:43|      1|
|2016-09-25 15:32:33|      3|
|2017-10-15 15:22:23|      4|
|2018-12-17 15:14:17|      4|
+-------------------+-------+



#### 22. second

In [18]:
# It will return the second part of the date.

df = (empdf
    .select("date")
    .withColumn("second", second("date")))
df.show()

+-------------------+------+
|               date|second|
+-------------------+------+
|2019-02-01 15:12:13|    13|
|  2018-04-01 5:12:3|     3|
|  2017-06-05 1:2:13|    13|
|2019-08-10 10:52:53|    53|
| 2016-01-11 5:52:43|    43|
|2015-04-14 19:32:33|    33|
|2019-02-21 15:42:43|    43|
|2016-09-25 15:32:33|    33|
|2017-10-15 15:22:23|    23|
|2018-12-17 15:14:17|    17|
+-------------------+------+



#### 23. to_date

In [19]:
# It will convert the String or TimeStamp to Date.

df = (empdf
    .select("date")
    .withColumn("to_date", to_date("date")))
df.show()
df.printSchema()

# Note : Check the data type of column "date" and "to-date".
# If the string format is 'yyyy-MM-dd HH:mm:ss' then we need not to specify the format. 
# Otherwise, specify the format as second arg in to_date function.

+-------------------+----------+
|               date|   to_date|
+-------------------+----------+
|2019-02-01 15:12:13|2019-02-01|
|  2018-04-01 5:12:3|2018-04-01|
|  2017-06-05 1:2:13|2017-06-05|
|2019-08-10 10:52:53|2019-08-10|
| 2016-01-11 5:52:43|2016-01-11|
|2015-04-14 19:32:33|2015-04-14|
|2019-02-21 15:42:43|2019-02-21|
|2016-09-25 15:32:33|2016-09-25|
|2017-10-15 15:22:23|2017-10-15|
|2018-12-17 15:14:17|2018-12-17|
+-------------------+----------+

root
 |-- date: string (nullable = true)
 |-- to_date: date (nullable = true)



#### 24. to_date with different date format

In [20]:
# Here we will convert the string of format 'dd/MM/yyyy HH:mm:ss' to "date" data type. Default format it 'yyyy-MM-dd`.
df1 = spark.createDataFrame([('15/02/2019 10:30:00',)], ['date'])
df2 = (df1
    .withColumn("new_date", to_date("date", 'dd/MM/yyyy HH:mm:ss')))
    
df2.show()

+-------------------+----------+
|               date|  new_date|
+-------------------+----------+
|15/02/2019 10:30:00|2019-02-15|
+-------------------+----------+



#### 25. to_timestamp

In [22]:
# It will convert the String to TimeStamp. Here we will convert the string of format 'dd/MM/yyyy HH:mm:ss' to "timestamp" data type.
# default format is 'yyyy-MM-dd HH:mm:ss'

df1 = spark.createDataFrame([('15/02/2019 10:30:00',)], ['date'])
df2 = (df1
    .withColumn("new_date", to_timestamp("date", 'dd/MM/yyyy HH:mm:ss')))
df2.show()
df2.printSchema()

+-------------------+-------------------+
|               date|           new_date|
+-------------------+-------------------+
|15/02/2019 10:30:00|2019-02-15 10:30:00|
+-------------------+-------------------+

root
 |-- date: string (nullable = true)
 |-- new_date: timestamp (nullable = true)



#### 26. to_utc_timestamp

In [23]:
# Convert given timestamp to UTC timestamp.
# For e.g. lets convert the "PST" timestamp to "UTC" timestamp.
df = (empdf
    .select("date")
    .withColumn("utc_timestamp", to_utc_timestamp("date", "PST")))
df.show()

+-------------------+-------------------+
|               date|      utc_timestamp|
+-------------------+-------------------+
|2019-02-01 15:12:13|2019-02-01 23:12:13|
|  2018-04-01 5:12:3|2018-04-01 12:12:03|
|  2017-06-05 1:2:13|2017-06-05 08:02:13|
|2019-08-10 10:52:53|2019-08-10 17:52:53|
| 2016-01-11 5:52:43|2016-01-11 13:52:43|
|2015-04-14 19:32:33|2015-04-15 02:32:33|
|2019-02-21 15:42:43|2019-02-21 23:42:43|
|2016-09-25 15:32:33|2016-09-25 22:32:33|
|2017-10-15 15:22:23|2017-10-15 22:22:23|
|2018-12-17 15:14:17|2018-12-17 23:14:17|
+-------------------+-------------------+



#### 27. weekofyear

In [24]:
# It will return the weekofyear for the given date.

df = (empdf
    .select("date")
    .withColumn("weekofyear", weekofyear("date")))
df.show()

+-------------------+----------+
|               date|weekofyear|
+-------------------+----------+
|2019-02-01 15:12:13|         5|
|  2018-04-01 5:12:3|        13|
+-------------------+----------+
only showing top 2 rows



#### 28. year

In [25]:
# It will return the year part of the date.

df = (empdf
    .select("date")
    .withColumn("year", year("date")))
df.show()

+-------------------+----+
|               date|year|
+-------------------+----+
|2019-02-01 15:12:13|2019|
|  2018-04-01 5:12:3|2018|
|  2017-06-05 1:2:13|2017|
|2019-08-10 10:52:53|2019|
| 2016-01-11 5:52:43|2016|
|2015-04-14 19:32:33|2015|
|2019-02-21 15:42:43|2019|
|2016-09-25 15:32:33|2016|
|2017-10-15 15:22:23|2017|
|2018-12-17 15:14:17|2018|
+-------------------+----+

